# Lab 06 Solution: Memory — Stateless vs Stateful

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatOllama(model="llama3.2:1b")

## Step 1: Without Memory

In [ ]:
r1 = llm.invoke([HumanMessage(content="My name is Priya and I work at UniGPS.")])
print(f"Turn 1 → {r1.content}\n")

r2 = llm.invoke([HumanMessage(content="What is my name and where do I work?")])
print(f"Turn 2 → {r2.content}")

## Step 2: With Memory

In [ ]:
conversation = [SystemMessage(content="You are a helpful assistant. Be concise.")]

for user_msg in [
    "My name is Priya and I work at UniGPS.",
    "What is my name and where do I work?",
    "Suggest a good lunch place near my office.",
]:
    conversation.append(HumanMessage(content=user_msg))
    response = llm.invoke(conversation)
    conversation.append(AIMessage(content=response.content))
    print(f"User: {user_msg}")
    print(f"AI:   {response.content}\n")

## Step 3: ChatBot Class with Memory

In [ ]:
class SimpleChatBot:
    def __init__(self, system_prompt: str, model: str = "llama3.2:1b"):
        self.llm = ChatOllama(model=model)
        self.history = [SystemMessage(content=system_prompt)]

    def chat(self, user_message: str) -> str:
        self.history.append(HumanMessage(content=user_message))
        response = self.llm.invoke(self.history)
        self.history.append(AIMessage(content=response.content))
        return response.content

    def get_history_length(self) -> int:
        return len(self.history)

    def clear_memory(self):
        self.history = [self.history[0]]

In [ ]:
bot = SimpleChatBot("You are a friendly travel assistant. Be concise \u2014 1-2 sentences.")
for msg in [
    "I want to visit Japan in March.",
    "What should I pack?",
    "Any must-see places?",
    "How much budget should I plan for a week?",
]:
    reply = bot.chat(msg)
    print(f"You: {msg}")
    print(f"Bot: {reply}")
    print(f"     [{bot.get_history_length()} messages]\n")

## Step 5: Sliding Window Memory

In [ ]:
class WindowedChatBot:
    def __init__(self, system_prompt: str, max_exchanges: int = 3):
        self.llm = ChatOllama(model="llama3.2:1b")
        self.system_msg = SystemMessage(content=system_prompt)
        self.history = []
        self.max_messages = max_exchanges * 2

    def chat(self, user_message: str) -> str:
        self.history.append(HumanMessage(content=user_message))
        if len(self.history) > self.max_messages:
            self.history = self.history[-self.max_messages:]
        messages = [self.system_msg] + self.history
        response = self.llm.invoke(messages)
        self.history.append(AIMessage(content=response.content))
        return response.content

In [ ]:
wbot = WindowedChatBot("You are a helpful assistant. Be concise.", max_exchanges=2)
for msg in [
    "My name is Raj.",
    "I love Python programming.",
    "I work at Google.",
    "What is my name?",
]:
    reply = wbot.chat(msg)
    print(f"You: {msg}")
    print(f"Bot: {reply}\n")

## TODO 1: PersistentChatBot with Save/Load (SOLUTION)

In [ ]:
import json as _json

class PersistentChatBot:
    """A chatbot that can save and load conversation memory to/from files."""
    
    def __init__(self, system_prompt: str, model: str = "llama3.2:1b"):
        self.llm = ChatOllama(model=model)
        self.history = [SystemMessage(content=system_prompt)]
    
    def chat(self, user_message: str) -> str:
        self.history.append(HumanMessage(content=user_message))
        response = self.llm.invoke(self.history)
        self.history.append(AIMessage(content=response.content))
        return response.content
    
    def get_history_length(self) -> int:
        return len(self.history)
    
    def save_memory(self, filepath: str):
        """Save conversation history to a JSON file."""
        data = []
        for msg in self.history:
            if isinstance(msg, SystemMessage):
                data.append({"role": "system", "content": msg.content})
            elif isinstance(msg, HumanMessage):
                data.append({"role": "human", "content": msg.content})
            elif isinstance(msg, AIMessage):
                data.append({"role": "ai", "content": msg.content})
        with open(filepath, "w") as f:
            _json.dump(data, f, indent=2)
    
    def load_memory(self, filepath: str):
        """Load conversation history from a JSON file."""
        with open(filepath) as f:
            data = _json.load(f)
        self.history = []
        for entry in data:
            role = entry["role"]
            content = entry["content"]
            if role == "system":
                self.history.append(SystemMessage(content=content))
            elif role == "human":
                self.history.append(HumanMessage(content=content))
            elif role == "ai":
                self.history.append(AIMessage(content=content))


# Demo: save and load across instances
import os
MEMFILE = "/tmp/k8s-lab-03-06/memory_test.json"
os.makedirs("/tmp/k8s-lab-03-06", exist_ok=True)

pbot = PersistentChatBot("You are a helpful assistant. Be concise.")
pbot.chat("My name is Priya and I love Kubernetes.")
pbot.chat("I work at BrainUpgrade.")
reply = pbot.chat("What do I love?")
print(f"Bot 1: {reply}")
print(f"History length: {pbot.get_history_length()}")

pbot.save_memory(MEMFILE)
print(f"\nMemory saved to {MEMFILE}")

# New bot loads the memory
pbot2 = PersistentChatBot("You are a helpful assistant. Be concise.")
pbot2.load_memory(MEMFILE)
print(f"Bot 2 loaded {pbot2.get_history_length()} messages")

reply2 = pbot2.chat("What is my name and where do I work?")
print(f"Bot 2: {reply2}")
print("Memory persists across bot instances!")

In [ ]:
# Validation
import os
score1 = 0
checks = []
MEMFILE = "/tmp/k8s-lab-03-06/memory_test.json"
os.makedirs("/tmp/k8s-lab-03-06", exist_ok=True)

# Test 1: Class works (can chat)
try:
    pbot = PersistentChatBot("You are a helpful assistant. Be concise.")
    reply = pbot.chat("My name is Priya and I love Kubernetes.")
    if isinstance(reply, str) and reply != "___" and len(reply) > 0:
        checks.append(("PersistentChatBot can chat", "PASS"))
        score1 += 1
    else:
        checks.append(("PersistentChatBot can chat", "FAIL"))
except Exception as e:
    checks.append((f"PersistentChatBot can chat ({e})", "FAIL"))

# Test 2: Save works
try:
    pbot.chat("I work at BrainUpgrade.")
    pbot.save_memory(MEMFILE)
    if os.path.exists(MEMFILE):
        with open(MEMFILE) as f:
            data = _json.load(f)
        if isinstance(data, list) and len(data) >= 3:
            checks.append(("save_memory creates valid JSON", "PASS"))
            score1 += 1
        else:
            checks.append((f"save_memory creates valid JSON (got {len(data) if isinstance(data, list) else type(data).__name__})", "FAIL"))
    else:
        checks.append(("save_memory creates valid JSON (file not created)", "FAIL"))
except Exception as e:
    checks.append((f"save_memory creates valid JSON ({e})", "FAIL"))

# Test 3: Load works
try:
    pbot2 = PersistentChatBot("You are a helpful assistant. Be concise.")
    pbot2.load_memory(MEMFILE)
    if pbot2.get_history_length() >= 3:
        checks.append(("load_memory restores history", "PASS"))
        score1 += 1
    else:
        checks.append((f"load_memory restores history (length={pbot2.get_history_length()})", "FAIL"))
except Exception as e:
    checks.append((f"load_memory restores history ({e})", "FAIL"))

# Test 4: Memory persists — new bot remembers old facts
try:
    reply2 = pbot2.chat("What is my name and where do I work?")
    # The bot should be able to reference Priya and BrainUpgrade from loaded memory
    if isinstance(reply2, str) and len(reply2) > 0 and reply2 != "___":
        checks.append(("Memory persists across bot instances", "PASS"))
        score1 += 1
    else:
        checks.append(("Memory persists across bot instances", "FAIL"))
except Exception as e:
    checks.append((f"Memory persists across bot instances ({e})", "FAIL"))

for check, status in checks:
    print(f"[{status}] {check}")
print(f"\nTODO 1 Score: {score1}/4")

## TODO 2: Summary Memory ChatBot

In [ ]:
class SummaryChatBot:
    def __init__(self, system_prompt: str, summarize_after: int = 6):
        self.llm = ChatOllama(model="llama3.2:1b")
        self.system_msg = SystemMessage(content=system_prompt)
        self.history = []
        self.summarize_after = summarize_after

    def chat(self, user_message: str) -> str:
        self.history.append(HumanMessage(content=user_message))

        # Summarize old messages if history is too long
        if len(self.history) > self.summarize_after:
            old_messages = self.history[:4]
            old_text = "\n".join(
                f"{'User' if isinstance(m, HumanMessage) else 'AI'}: {m.content}"
                for m in old_messages
            )
            summary = self.llm.invoke([
                SystemMessage(content="Summarize this conversation in 2 sentences. Preserve key facts."),
                HumanMessage(content=old_text),
            ]).content
            self.history = [HumanMessage(content=f"[Earlier conversation summary: {summary}]")] + self.history[4:]
            print(f"  [Summarized! History compressed to {len(self.history)} messages]")

        messages = [self.system_msg] + self.history
        response = self.llm.invoke(messages)
        self.history.append(AIMessage(content=response.content))
        return response.content

In [ ]:
sbot = SummaryChatBot("You are a helpful assistant. Be concise.", summarize_after=6)
for msg in [
    "My favorite number is 42.",
    "I'm learning Python for data science.",
    "I work at UniGPS in Bangalore.",
    "What's a good Python library for data analysis?",
    "Can you remind me of my favorite number?",
]:
    reply = sbot.chat(msg)
    print(f"You: {msg}")
    print(f"Bot: {reply}\n")

## TODO 3: Memory Limit Comparison

In [ ]:
full_bot = SimpleChatBot("You are a helpful assistant. Be concise.")
win_bot = WindowedChatBot("You are a helpful assistant. Be concise.", max_exchanges=3)

full_bot.chat("My favorite number is 42.")
win_bot.chat("My favorite number is 42.")

for msg in [
    "Tell me about Python.",
    "What is Flask??",
    "Explain FastAPI.",
    "What is REST API?",
]:
    full_bot.chat(msg)
    win_bot.chat(msg)

full_answer = full_bot.chat("What is my favorite number?")
win_answer = win_bot.chat("What is my favorite number?")

print(f"Full memory bot: {full_answer}")
print(f"Window (3) bot:  {win_answer}")
print(f"\nFull memory keeps everything \u2014 window forgets old messages!")